# 第2章: 確率分布と母関数

## 学習目標
- 確率分布の基本概念を理解する
- 確率質量関数と確率密度関数を区別できる
- 母関数（積率母関数、確率母関数、特性関数）の定義と性質を理解する
- 母関数を用いた計算ができるようになる

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import factorial
import seaborn as sns

plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
np.random.seed(42)

## 2.1 確率変数と確率分布

### キーコンセプト

**確率変数 (Random Variable)**: 標本空間から実数への関数 $X: \Omega \to \mathbb{R}$

### 離散型確率変数
- **確率質量関数 (PMF)**: $p(x) = P(X = x)$
- **累積分布関数 (CDF)**: $F(x) = P(X \leq x) = \sum_{x_i \leq x} p(x_i)$

### 連続型確率変数
- **確率密度関数 (PDF)**: $f(x)$ where $P(a \leq X \leq b) = \int_a^b f(x)dx$
- **累積分布関数 (CDF)**: $F(x) = P(X \leq x) = \int_{-\infty}^x f(t)dt$

In [ ]:
# 離散型と連続型の比較
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 離散型: ポアソン分布
lam = 5
x_discrete = np.arange(0, 15)
pmf = stats.poisson.pmf(x_discrete, lam)
cdf_discrete = stats.poisson.cdf(x_discrete, lam)

axes[0, 0].bar(x_discrete, pmf, color='steelblue', alpha=0.7)
axes[0, 0].set_title(f'PMF: Poisson(λ={lam})', fontsize=14)
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('P(X = x)')

axes[0, 1].step(x_discrete, cdf_discrete, where='mid', color='steelblue', linewidth=2)
axes[0, 1].scatter(x_discrete, cdf_discrete, color='steelblue', s=50, zorder=5)
axes[0, 1].set_title(f'CDF: Poisson(λ={lam})', fontsize=14)
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('P(X ≤ x)')

# 連続型: 正規分布
mu, sigma = 5, 1.5
x_continuous = np.linspace(0, 10, 200)
pdf = stats.norm.pdf(x_continuous, mu, sigma)
cdf_continuous = stats.norm.cdf(x_continuous, mu, sigma)

axes[1, 0].plot(x_continuous, pdf, color='coral', linewidth=2)
axes[1, 0].fill_between(x_continuous, pdf, alpha=0.3, color='coral')
axes[1, 0].set_title(f'PDF: Normal(μ={mu}, σ={sigma})', fontsize=14)
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('f(x)')

axes[1, 1].plot(x_continuous, cdf_continuous, color='coral', linewidth=2)
axes[1, 1].set_title(f'CDF: Normal(μ={mu}, σ={sigma})', fontsize=14)
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('F(x)')

plt.tight_layout()
plt.show()

## 2.2 積率母関数 (Moment Generating Function)

### 定義
$$M_X(t) = E[e^{tX}]$$

- 離散型: $M_X(t) = \sum_x e^{tx} p(x)$
- 連続型: $M_X(t) = \int_{-\infty}^{\infty} e^{tx} f(x) dx$

### 重要な性質
1. **積率の計算**: $E[X^n] = M_X^{(n)}(0) = \frac{d^n}{dt^n} M_X(t) \big|_{t=0}$
2. **一意性**: MGFが存在すれば、分布を一意に決定
3. **独立変数の和**: $X, Y$が独立なら $M_{X+Y}(t) = M_X(t) \cdot M_Y(t)$
4. **線形変換**: $M_{aX+b}(t) = e^{bt} M_X(at)$

In [ ]:
# 主要分布のMGF
print("主要な分布の積率母関数")
print("="*60)
print("\n1. ベルヌーイ分布 Ber(p):")
print("   M_X(t) = 1 - p + p*e^t")
print("\n2. 二項分布 Bin(n, p):")
print("   M_X(t) = (1 - p + p*e^t)^n")
print("\n3. ポアソン分布 Poi(λ):")
print("   M_X(t) = exp(λ(e^t - 1))")
print("\n4. 指数分布 Exp(λ):")
print("   M_X(t) = λ/(λ - t), t < λ")
print("\n5. 正規分布 N(μ, σ²):")
print("   M_X(t) = exp(μt + σ²t²/2)")
print("\n6. ガンマ分布 Gamma(α, β):")
print("   M_X(t) = (1 - t/β)^(-α), t < β")

In [ ]:
# MGFから積率を計算する例：正規分布
import sympy as sp

t, mu, sigma = sp.symbols('t mu sigma', real=True)
sigma = sp.symbols('sigma', positive=True)

# 正規分布のMGF
M = sp.exp(mu*t + sigma**2 * t**2 / 2)

# 1次微分（期待値）
M_prime = sp.diff(M, t)
E_X = M_prime.subs(t, 0)

# 2次微分
M_double_prime = sp.diff(M, t, 2)
E_X2 = M_double_prime.subs(t, 0)

print("正規分布 N(μ, σ²) のMGFから積率を計算")
print("="*50)
print(f"M_X(t) = {M}")
print(f"\nM'_X(t) = {M_prime}")
print(f"E[X] = M'_X(0) = {E_X}")
print(f"\nM''_X(t) = {sp.simplify(M_double_prime)}")
print(f"E[X²] = M''_X(0) = {E_X2}")
print(f"\nVar(X) = E[X²] - E[X]² = {sp.simplify(E_X2 - E_X**2)}")

In [ ]:
# MGFの可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

t_vals = np.linspace(-2, 2, 100)

# ポアソン分布のMGF
for lam in [1, 3, 5]:
    mgf_poisson = np.exp(lam * (np.exp(t_vals) - 1))
    axes[0].plot(t_vals, mgf_poisson, label=f'λ={lam}', linewidth=2)
axes[0].set_title('MGF of Poisson Distribution', fontsize=14)
axes[0].set_xlabel('t')
axes[0].set_ylabel('M_X(t)')
axes[0].legend()
axes[0].set_ylim([0, 50])
axes[0].grid(True, alpha=0.3)

# 正規分布のMGF
for mu, sigma in [(0, 1), (0, 2), (1, 1)]:
    mgf_normal = np.exp(mu * t_vals + sigma**2 * t_vals**2 / 2)
    axes[1].plot(t_vals, mgf_normal, label=f'μ={mu}, σ={sigma}', linewidth=2)
axes[1].set_title('MGF of Normal Distribution', fontsize=14)
axes[1].set_xlabel('t')
axes[1].set_ylabel('M_X(t)')
axes[1].legend()
axes[1].set_ylim([0, 10])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.3 確率母関数 (Probability Generating Function)

### 定義（非負整数値の確率変数）
$$G_X(s) = E[s^X] = \sum_{k=0}^{\infty} s^k P(X = k)$$

### 重要な性質
1. **確率の復元**: $P(X = k) = \frac{G_X^{(k)}(0)}{k!}$
2. **期待値**: $E[X] = G_X'(1)$
3. **分散**: $Var(X) = G_X''(1) + G_X'(1) - [G_X'(1)]^2$
4. **独立変数の和**: $G_{X+Y}(s) = G_X(s) \cdot G_Y(s)$

In [ ]:
# 確率母関数の例
print("主要な離散分布の確率母関数")
print("="*60)
print("\n1. ベルヌーイ分布 Ber(p):")
print("   G_X(s) = 1 - p + ps")
print("\n2. 二項分布 Bin(n, p):")
print("   G_X(s) = (1 - p + ps)^n")
print("\n3. ポアソン分布 Poi(λ):")
print("   G_X(s) = exp(λ(s - 1))")
print("\n4. 幾何分布 Geo(p):")
print("   G_X(s) = ps/(1 - (1-p)s)")
print("\n5. 負の二項分布 NB(r, p):")
print("   G_X(s) = (ps/(1 - (1-p)s))^r")

In [ ]:
# PGFから期待値と分散を計算：ポアソン分布
s, lam_sym = sp.symbols('s lambda', positive=True)

# ポアソン分布のPGF
G = sp.exp(lam_sym * (s - 1))

G_prime = sp.diff(G, s)
G_double_prime = sp.diff(G, s, 2)

E_X = G_prime.subs(s, 1)
Var_X = G_double_prime.subs(s, 1) + G_prime.subs(s, 1) - G_prime.subs(s, 1)**2

print("ポアソン分布 Poi(λ) のPGFから統計量を計算")
print("="*50)
print(f"G_X(s) = {G}")
print(f"G'_X(s) = {G_prime}")
print(f"\nE[X] = G'_X(1) = {E_X}")
print(f"Var(X) = {sp.simplify(Var_X)}")

## 2.4 特性関数 (Characteristic Function)

### 定義
$$\phi_X(t) = E[e^{itX}]$$

ここで $i$ は虚数単位。

### 重要な性質
1. **常に存在**: 全ての確率変数に対して定義可能
2. **有界**: $|\phi_X(t)| \leq 1$
3. **積率**: $E[X^n] = \frac{1}{i^n} \phi_X^{(n)}(0)$
4. **一意性**: 特性関数は分布を一意に決定
5. **反転公式が存在**

In [ ]:
# 特性関数の可視化：正規分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

t_vals = np.linspace(-5, 5, 200)

# 正規分布 N(0, σ²) の特性関数: exp(-σ²t²/2)
for sigma in [0.5, 1, 2]:
    cf_real = np.exp(-sigma**2 * t_vals**2 / 2)  # 実部（虚部は0）
    axes[0].plot(t_vals, cf_real, label=f'σ={sigma}', linewidth=2)

axes[0].set_title('Characteristic Function of N(0, σ²) - Real Part', fontsize=14)
axes[0].set_xlabel('t')
axes[0].set_ylabel('Re[φ(t)]')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ポアソン分布の特性関数: exp(λ(e^(it) - 1))
for lam in [1, 3, 5]:
    cf_complex = np.exp(lam * (np.exp(1j * t_vals) - 1))
    axes[1].plot(t_vals, np.abs(cf_complex), label=f'λ={lam}', linewidth=2)

axes[1].set_title('Characteristic Function of Poisson(λ) - Absolute Value', fontsize=14)
axes[1].set_xlabel('t')
axes[1].set_ylabel('|φ(t)|')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.5 母関数の応用：独立な確率変数の和

独立な確率変数の和の分布を求める際、母関数が非常に有用です。

### 定理
$X$ と $Y$ が独立なとき:
- $M_{X+Y}(t) = M_X(t) \cdot M_Y(t)$
- $G_{X+Y}(s) = G_X(s) \cdot G_Y(s)$
- $\phi_{X+Y}(t) = \phi_X(t) \cdot \phi_Y(t)$

In [ ]:
# 独立なポアソン変数の和がポアソンになることをシミュレーション
n_samples = 100000

lam1, lam2 = 3, 5
X = np.random.poisson(lam1, n_samples)
Y = np.random.poisson(lam2, n_samples)
Z = X + Y

# 理論的にはZ ~ Poi(λ1 + λ2)
fig, ax = plt.subplots(figsize=(10, 6))

# ヒストグラム
max_val = max(Z) + 1
bins = np.arange(-0.5, max_val + 0.5, 1)
ax.hist(Z, bins=bins, density=True, alpha=0.7, label='Simulation X+Y')

# 理論分布
x_vals = np.arange(0, max_val)
theoretical = stats.poisson.pmf(x_vals, lam1 + lam2)
ax.plot(x_vals, theoretical, 'ro-', markersize=8, label=f'Poi({lam1}+{lam2})')

ax.set_title(f'Sum of Independent Poisson: Poi({lam1}) + Poi({lam2})', fontsize=14)
ax.set_xlabel('Value')
ax.set_ylabel('Probability')
ax.legend()
plt.show()

print(f"理論平均: {lam1 + lam2}, シミュレーション平均: {np.mean(Z):.4f}")
print(f"理論分散: {lam1 + lam2}, シミュレーション分散: {np.var(Z):.4f}")

In [ ]:
# 独立な正規変数の和
n_samples = 100000

mu1, sigma1 = 2, 1
mu2, sigma2 = 3, 1.5

X = np.random.normal(mu1, sigma1, n_samples)
Y = np.random.normal(mu2, sigma2, n_samples)
Z = X + Y

# 理論的には Z ~ N(μ1+μ2, σ1²+σ2²)
mu_sum = mu1 + mu2
sigma_sum = np.sqrt(sigma1**2 + sigma2**2)

fig, ax = plt.subplots(figsize=(10, 6))

# ヒストグラム
ax.hist(Z, bins=50, density=True, alpha=0.7, label='Simulation X+Y')

# 理論分布
x_vals = np.linspace(min(Z), max(Z), 200)
theoretical = stats.norm.pdf(x_vals, mu_sum, sigma_sum)
ax.plot(x_vals, theoretical, 'r-', linewidth=2, 
        label=f'N({mu_sum}, {sigma_sum**2:.2f})')

ax.set_title(f'Sum of Independent Normals: N({mu1},{sigma1**2}) + N({mu2},{sigma2**2})', 
             fontsize=14)
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.legend()
plt.show()

## 2.6 キュムラント母関数

### 定義
$$K_X(t) = \log M_X(t)$$

### キュムラント
$$\kappa_n = K_X^{(n)}(0)$$

- $\kappa_1 = E[X]$ (平均)
- $\kappa_2 = Var(X)$ (分散)
- $\kappa_3$ (歪度に関連)
- $\kappa_4$ (尖度に関連)

### 重要な性質
独立な確率変数の和のキュムラントは、各キュムラントの和:
$$K_{X+Y}(t) = K_X(t) + K_Y(t)$$

In [ ]:
# キュムラントの計算例：正規分布
t_sym = sp.symbols('t')
mu_sym, sigma_sym = sp.symbols('mu sigma', real=True)

# 正規分布のMGF
M_normal = sp.exp(mu_sym * t_sym + sigma_sym**2 * t_sym**2 / 2)
K_normal = sp.log(M_normal)

print("正規分布のキュムラント母関数")
print("="*50)
print(f"K_X(t) = log(M_X(t)) = {K_normal}")
print(f"\nキュムラント:")
print(f"κ₁ = K'(0) = {sp.diff(K_normal, t_sym).subs(t_sym, 0)}")
print(f"κ₂ = K''(0) = {sp.diff(K_normal, t_sym, 2).subs(t_sym, 0)}")
print(f"κ₃ = K'''(0) = {sp.diff(K_normal, t_sym, 3).subs(t_sym, 0)}")
print(f"κ₄ = K''''(0) = {sp.diff(K_normal, t_sym, 4).subs(t_sym, 0)}")
print("\n正規分布では κ₃ = κ₄ = ... = 0（3次以上のキュムラントが0）")

## 2.7 練習問題

### 問題1
二項分布 $Bin(n, p)$ の積率母関数を用いて、期待値 $E[X]$ と分散 $Var(X)$ を求めよ。

### 問題2
$X \sim Poi(\lambda_1)$、$Y \sim Poi(\lambda_2)$ が独立のとき、$X + Y$ の分布を母関数を用いて求めよ。

### 問題3
指数分布 $Exp(\lambda)$ の特性関数を求め、それを用いて期待値と分散を計算せよ。

In [ ]:
# 問題1の解答
print("問題1の解答: 二項分布のMGFから期待値と分散")
print("="*50)

t_sym = sp.symbols('t')
n_sym, p_sym = sp.symbols('n p', positive=True)

# 二項分布のMGF: M(t) = (1 - p + p*e^t)^n
M_binom = (1 - p_sym + p_sym * sp.exp(t_sym))**n_sym

# 1次微分
M_prime = sp.diff(M_binom, t_sym)
E_X = sp.simplify(M_prime.subs(t_sym, 0))

# 2次微分
M_double_prime = sp.diff(M_binom, t_sym, 2)
E_X2 = sp.simplify(M_double_prime.subs(t_sym, 0))

# 分散
Var_X = sp.simplify(E_X2 - E_X**2)

print(f"M_X(t) = {M_binom}")
print(f"\nE[X] = {E_X}")
print(f"E[X²] = {E_X2}")
print(f"Var(X) = E[X²] - E[X]² = {Var_X}")

In [ ]:
# 問題2の解答
print("問題2の解答: ポアソン変数の和")
print("="*50)

print("\nポアソン分布のMGF: M_X(t) = exp(λ(e^t - 1))")
print("\nX ~ Poi(λ₁), Y ~ Poi(λ₂) が独立のとき、")
print("M_{X+Y}(t) = M_X(t) × M_Y(t)")
print("         = exp(λ₁(e^t - 1)) × exp(λ₂(e^t - 1))")
print("         = exp((λ₁ + λ₂)(e^t - 1))")
print("\nこれは Poi(λ₁ + λ₂) のMGFなので、")
print("X + Y ~ Poi(λ₁ + λ₂)")

In [ ]:
# 問題3の解答
print("問題3の解答: 指数分布の特性関数")
print("="*50)

t_sym = sp.symbols('t', real=True)
lam_sym = sp.symbols('lambda', positive=True)
i = sp.I

# 指数分布のCF: φ(t) = λ/(λ - it)
phi = lam_sym / (lam_sym - i * t_sym)

# 微分して積率を計算
phi_prime = sp.diff(phi, t_sym)
phi_double_prime = sp.diff(phi, t_sym, 2)

# E[X] = φ'(0)/i
E_X = sp.simplify(phi_prime.subs(t_sym, 0) / i)

# E[X²] = φ''(0)/i²
E_X2 = sp.simplify(phi_double_prime.subs(t_sym, 0) / (i**2))

# 分散
Var_X = sp.simplify(E_X2 - E_X**2)

print(f"φ_X(t) = {phi}")
print(f"\nφ'_X(t) = {sp.simplify(phi_prime)}")
print(f"E[X] = φ'(0)/i = {E_X}")
print(f"\nE[X²] = φ''(0)/i² = {E_X2}")
print(f"Var(X) = {Var_X}")